In [16]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
===============================================================================
JCP FINAL v6.0.3 可视化脚本 —— SCIENCE 期刊级
✅ 已修复：路径、checkpoint 文件名、数据读取
===============================================================================
"""
import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("matplotlib").setLevel(logging.ERROR)
logging.getLogger("fontTools").setLevel(logging.WARNING)

import os
import json
import csv
import numpy as np
import torch
from collections import defaultdict
from datetime import datetime

import matplotlib as mpl
mpl.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.gridspec import GridSpec
import matplotlib.patches as mpatches

from scipy.ndimage import uniform_filter1d
from scipy.stats import ttest_rel


# ==============================================================================
# ✅ 路径已修改为你当前运行成功的版本
# ==============================================================================
RESULT_DIR = "jcp_results_v6_0_3"
FIG_DIR = os.path.join(RESULT_DIR, "figures_science")
TABLE_DIR = os.path.join(RESULT_DIR, "paper_tables")
CHECKPOINT_DIR = os.path.join(RESULT_DIR, "checkpoints")
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)

PAPER_DATA_FILE = os.path.join(TABLE_DIR, "paper_key_findings.txt")

METHOD_KEYS = ["Vanilla", "Penalty", "Fixed-λ", "SA-PINN-pos", "TwoStage-Proj"]
METHOD_LABELS = ["Vanilla", "Penalty", "Fixed-λ", "SA-PINN", "Ours"]
OURS = "TwoStage-Proj"
PDE_LIST = ["KdV", "Burgers", "NLS", "KS"]
PDE_TITLES = ["Korteweg-de Vries", "Burgers", "Nonlinear Schrödinger", "Kuramoto-Sivashinsky"]

ITERATIONS = 2000
WARMUP_ITERS = 60

COLORS = {
    "Vanilla": "#D62728",
    "Penalty": "#FF7F0E",
    "Fixed-λ": "#2CA02C",
    "SA-PINN-pos": "#1F77B4",
    "TwoStage-Proj": "#000000",
}

BASELINE_GRAY = "#CCCCCC"

WIDTH_SINGLE = 3.4
WIDTH_DOUBLE = 7.0
WIDTH_FULL = 7.0

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica"],
    "mathtext.fontset": "stix",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "legend.frameon": True,
    "legend.framealpha": 0.95,
    "legend.edgecolor": "black",
    "legend.fancybox": False,
    "axes.linewidth": 0.7,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.width": 0.6,
    "xtick.major.size": 3,
    "ytick.major.width": 0.6,
    "ytick.major.size": 3,
    "grid.alpha": 0.25,
    "grid.linewidth": 0.4,
    "axes.grid": True,
    "axes.axisbelow": True,
    "figure.facecolor": "white",
    "savefig.dpi": 1000,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


# ==============================================================================
# 工具函数
# ==============================================================================
def style_for_method(m):
    if m == OURS:
        return dict(color=COLORS[m], lw=2.0, alpha=1.0, zorder=10, linestyle="-", marker="o", markersize=2, markevery=20)
    else:
        return dict(color=COLORS[m], lw=1.2, alpha=0.8, zorder=3, linestyle="-", marker="s", markersize=1.5, markevery=30)

def savefig(fig, name, formats=["pdf", "png", "eps"]):
    for fmt in formats:
        if fmt == "pdf":
            fig.savefig(os.path.join(FIG_DIR, f"{name}.pdf"), transparent=True, pad_inches=0.02, bbox_inches="tight", dpi=1000)
        elif fmt == "png":
            fig.savefig(os.path.join(FIG_DIR, f"{name}.png"), transparent=False, pad_inches=0.02, bbox_inches="tight", dpi=1000)
        elif fmt == "eps":
            fig.savefig(os.path.join(FIG_DIR, f"{name}.eps"), transparent=True, pad_inches=0.02, bbox_inches="tight", dpi=1000)
    plt.close(fig)
    print(f"  ✓ {name}.pdf/.png/.eps")

def add_panel_label(ax, txt, x=-0.18, y=1.08, fontsize=9):
    ax.text(x, y, txt, transform=ax.transAxes, fontsize=fontsize, fontweight="bold", va="top", ha="left")

def plot_warmup(ax, warmup_end=WARMUP_ITERS):
    ax.axvline(warmup_end, color="0.5", ls=":", lw=0.8, zorder=2)
    ax.text(warmup_end + 10, ax.get_ylim()[1] * 0.1, "Warmup", fontsize=7, color="0.4")

def smooth(arr, w=20):
    return uniform_filter1d(np.asarray(arr, dtype=float), size=w, mode="nearest")

def load_json(fname):
    path = os.path.join(RESULT_DIR, fname)
    if not os.path.exists(path): return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def robust_log_ylim(y_all, pad_decades=0.5, floor=1e-16):
    y_all = np.asarray(y_all)
    valid = y_all[(y_all > 0) & np.isfinite(y_all)]
    if len(valid) == 0: return 1e-8, 1e0
    ymin = max(np.min(valid), floor)
    ymax = np.max(valid)
    log_ymin = np.floor(np.log10(ymin)) - pad_decades
    log_ymax = np.ceil(np.log10(ymax)) + 0.3
    return 10**log_ymin, 10**log_ymax

# ==============================================================================
# ✅ 已修复：checkpoint 文件名
# ==============================================================================
def load_all_loss_data():
    all_losses = {}
    for pde in PDE_LIST:
        all_losses[pde] = {}
        for method in METHOD_KEYS:
            method_losses = defaultdict(list)
            for seed in range(42, 47):
                T_suffix = "_T1p00"
                ckpt_path = os.path.join(CHECKPOINT_DIR, f"ckpt_{pde}_{method}{T_suffix}_seed{seed}.pth")
                if not os.path.exists(ckpt_path):
                    continue
                try:
                    ckpt = torch.load(ckpt_path, map_location="cpu")
                    loss_rec = ckpt["loss_rec"]
                    for key in ["Lp", "Lic", "Cinv", "Linv", "Ltot", "lam_ic", "lam_inv"]:
                        if key in loss_rec and len(loss_rec[key]) == ITERATIONS:
                            method_losses[key].append(loss_rec[key])
                except Exception as e:
                    pass
            avg_losses = {}
            for key in method_losses:
                if method_losses[key]:
                    avg_losses[key] = np.mean(method_losses[key], axis=0)
            all_losses[pde][method] = avg_losses
    return all_losses

# ==============================================================================
# ✅ 已修复：守恒曲线字段读取
# ==============================================================================
def load_all_conservation_data():
    all_conservation = {}
    for pde in PDE_LIST:
        all_conservation[pde] = {}
        for method in METHOD_KEYS:
            pred_path = os.path.join(RESULT_DIR, "fields", f"pred_{pde}_{method}.npz")
            if not os.path.exists(pred_path):
                continue
            try:
                data = np.load(pred_path)
                x = data["x"]
                t = data["t"]
                dx = x[1] - x[0]

                if "u_proj" in data.files:
                    u = data["u_proj"]
                elif "u_raw" in data.files:
                    u = data["u_raw"]
                elif "u" in data.files:
                    u = data["u"]
                else:
                    continue

                conservation = []
                c = []
                if pde == "KdV":
                    for i in range(len(t)):
                        u_t = u[i, :, 0]
                        ux_t = np.gradient(u_t, dx)
                        I_t = np.trapz(0.5 * ux_t**2 - u_t**3, x)
                        conservation.append(I_t)
                    I0 = conservation[0]
                    c = np.abs((np.array(conservation) - I0) / abs(I0))
                elif pde == "NLS":
                    for i in range(len(t)):
                        u_t = u[i, :, :]
                        I_t = np.trapz(u_t[:, 0]**2 + u_t[:, 1]**2, x)
                        conservation.append(I_t)
                    I0 = conservation[0]
                    c = np.abs((np.array(conservation) - I0) / abs(I0))
                elif pde == "KS":
                    for i in range(len(t)):
                        u_t = u[i, :, 0]
                        I_t = np.trapz(u_t, x)
                        conservation.append(I_t)
                    I0 = conservation[0]
                    c = np.abs((np.array(conservation) - I0) / abs(I0))
                elif pde == "Burgers":
                    E_prev = 0.0
                    c = []
                    for i in range(len(t)):
                        u_t = u[i, :, 0]
                        E_t = 0.5 * np.trapz(u_t**2, x)
                        ux_t = np.gradient(u_t, dx)
                        D_t = 0.01 * np.trapz(ux_t**2, x)
                        if i == 0:
                            E0 = E_t
                            c.append(0.0)
                        else:
                            dt = t[i] - t[i-1]
                            dEdt = (E_t - E_prev) / dt
                            c.append(abs(dEdt + D_t) / abs(E0))
                        E_prev = E_t
                    c = np.array(c)
                all_conservation[pde][method] = {"t": t, "c": c}
            except Exception as e:
                continue
    return all_conservation

# ==============================================================================
# 绘图函数
# ==============================================================================
def fig1_conservation():
    print("\n[Fig 1] 绘制守恒性对比...")
    curves = load_all_conservation_data()
    if not curves:
        print("  ⚠ 无数据")
        return
    fig = plt.figure(figsize=(WIDTH_FULL, 4.5))
    gs = GridSpec(2, 2, figure=fig, hspace=0.35, wspace=0.3, left=0.08, right=0.98, top=0.92, bottom=0.12)
    for i, pde in enumerate(PDE_LIST):
        ax = fig.add_subplot(gs[i // 2, i % 2])
        if pde not in curves:
            ax.axis('off')
            continue
        y_all = []
        for m in METHOD_KEYS:
            if m not in curves[pde]: continue
            d = curves[pde][m]
            t = np.array(d["t"])
            y = np.abs(np.array(d["c"]))
            downsample_rate = 10
            t, y = t[::downsample_rate], y[::downsample_rate]
            y_all.extend(y)
            ax.semilogy(t, y, **style_for_method(m), label=METHOD_LABELS[METHOD_KEYS.index(m)])
        if not y_all: continue
        ax.set_xlim(0, 1.0)
        ax.set_ylim(robust_log_ylim(y_all))
        ax.set_title(PDE_TITLES[i], fontweight="bold")
        ax.set_xlabel(r"Time $t$")
        ax.set_ylabel(r"$|\Delta I(t)|/I_0$")
        ax.grid(True, alpha=0.2)
        add_panel_label(ax, f"({chr(97+i)})", x=-0.20)
    h, l = fig.axes[0].get_legend_handles_labels()
    if h:
        fig.legend(h, l, loc="upper center", ncol=5, frameon=True, fontsize=7, bbox_to_anchor=(0.5, 0.98))
    savefig(fig, "fig1_conservation_main")

def fig2_pareto_front(table):
    print("\n[Fig 2] 绘制Pareto前沿...")
    if not table: return
    fig = plt.figure(figsize=(WIDTH_FULL, 4.5))
    gs = GridSpec(2,2, figure=fig, hspace=0.35, wspace=0.3, left=0.08, right=0.98, top=0.92, bottom=0.12)
    for i,pde in enumerate(PDE_LIST):
        ax = fig.add_subplot(gs[i//2,i%2])
        if pde not in table: continue
        x_vals,y_vals,ms,cs = [],[],[],[]
        for m in METHOD_KEYS:
            if m not in table[pde]: continue
            try:
                data=table[pde][m]
                x=data.get("mean_l2_raw", data.get("mean_l2", 0))
                y=data.get("mean_constraint_raw", data.get("mean_constraint", 0))
                if x<=0 or y<=0: continue
                x_vals.append(x)
                y_vals.append(y)
                ms.append(100 if m==OURS else 50)
                cs.append(COLORS[m])
            except: continue
        for x,y,m,c in zip(x_vals,y_vals,ms,cs):
            ax.scatter(x,y,s=m,c=c,marker="o" if m!=100 else "*",zorder=10 if m==100 else 3,edgecolors="black",linewidth=0.5)
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_title(PDE_TITLES[i],fontweight="bold")
        ax.set_xlabel(r"$L^2$ Error")
        ax.set_ylabel(r"Constraint Violation")
        add_panel_label(ax,f"({chr(97+i)})",x=-0.20)
    savefig(fig,"fig2_pareto_frontier")

def fig3_projection_l2(table):
    print("\n[Fig3] 投影L2对比...")
    valid=[p for p in PDE_LIST if p!="Burgers" and p in table and OURS in table[p]]
    if not valid: return
    fig,axes=plt.subplots(1,len(valid),figsize=(WIDTH_FULL,2.8))
    axes=np.atleast_1d(axes).flatten()
    for i,pde in enumerate(valid):
        ax=axes[i]
        data=table[pde][OURS]
        l2_raw=data.get("mean_l2_raw", data.get("mean_l2", 0))
        l2_proj=data.get("mean_l2_proj", l2_raw)
        ax.bar([0,1],[l2_raw,l2_proj],color=[COLORS[OURS],BASELINE_GRAY],width=0.5,edgecolor="black")
        ax.set_yscale("log")
        ax.set_title(pde,fontweight="bold")
        ax.set_xticks([0,1])
        ax.set_xticklabels(["Raw","Proj"])
        add_panel_label(ax,f"({chr(97+i)})",x=-0.25)
    fig.tight_layout()
    savefig(fig,"fig3_projection_l2")

def fig4_projection_residual(table):
    print("\n[Fig4] 投影残差对比...")
    valid=[p for p in PDE_LIST if p!="Burgers" and p in table and OURS in table[p]]
    if not valid: return
    fig,axes=plt.subplots(1,len(valid),figsize=(WIDTH_FULL,2.8))
    axes=np.atleast_1d(axes).flatten()
    for i,pde in enumerate(valid):
        ax=axes[i]
        data=table[pde][OURS]
        r_raw=data.get("mean_residual_raw", data.get("mean_resid", 0))
        r_proj=data.get("mean_residual_proj", r_raw)
        ax.bar([0,1],[r_raw,r_proj],color=[COLORS[OURS],BASELINE_GRAY],width=0.5,edgecolor="black")
        ax.set_yscale("log")
        ax.set_title(pde,fontweight="bold")
        ax.set_xticks([0,1])
        ax.set_xticklabels(["Raw","Proj"])
        add_panel_label(ax,f"({chr(97+i)})",x=-0.25)
    fig.tight_layout()
    savefig(fig,"fig4_projection_residual")

def fig5_projection_params(table):
    print("\n[Fig5] 投影参数...")
    valid=[p for p in PDE_LIST if p!="Burgers" and p in table and OURS in table[p]]
    if not valid: return
    fig,axes=plt.subplots(1,len(valid),figsize=(WIDTH_FULL,2.8))
    axes=np.atleast_1d(axes).flatten()
    for i,pde in enumerate(valid):
        ax=axes[i]
        data=table[pde][OURS]
        if pde in ["KdV","KS"]:
            me=data.get("mean_abs_alpha",0)
            ma=data.get("max_abs_alpha",0)
        else:
            me=data.get("mean_abs_s_minus_1",0)
            ma=data.get("max_abs_s_minus_1",0)
        ax.bar([0,1],[me,ma],color=[COLORS[OURS],BASELINE_GRAY],width=0.5,edgecolor="black")
        ax.set_title(pde,fontweight="bold")
        ax.set_xticks([0,1])
        ax.set_xticklabels(["Mean","Max"])
        add_panel_label(ax,f"({chr(97+i)})",x=-0.25)
    fig.tight_layout()
    savefig(fig,"fig5_projection_params")

def fig6_training_loss_kdv(all_losses):
    print("\n[Fig6] KdV损失...")
    if "KdV" not in all_losses: return
    fig=plt.figure(figsize=(WIDTH_FULL,4.5))
    gs=GridSpec(2,2,figure=fig,hspace=0.35,wspace=0.3)
    keys=[("Lp",r"$L_p$"),("Lic",r"$L_{ic}$"),("Cinv",r"$C_{\rm inv}$"),("Ltot",r"$L_{\rm tot}$")]
    for i,(k,lab) in enumerate(keys):
        ax=fig.add_subplot(gs[i//2,i%2])
        for m in METHOD_KEYS:
            try:
                y=smooth(all_losses["KdV"][m][k])
                ax.semilogy(y,**style_for_method(m))
            except: continue
        plot_warmup(ax)
        ax.set_title(lab,fontweight="bold")
        add_panel_label(ax,f"({chr(97+i)})")
    savefig(fig,"fig6_training_loss_kdv")

def fig8_training_cost(table):
    print("\n[Fig8] 训练成本...")
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(WIDTH_FULL,3.5))
    x=np.arange(len(METHOD_KEYS))
    tms,mms=[],[]
    for m in METHOD_KEYS:
        try:
            t=np.mean([table[p][m]["train_time_mean"] for p in PDE_LIST if p in table and m in table[p]])
            m=np.mean([table[p][m]["peak_gpu_mean"] for p in PDE_LIST if p in table and m in table[p]])
        except:
            t=m=0
        tms.append(t)
        mms.append(m)
    ax1.bar(x,tms,color=[COLORS[m] for m in METHOD_KEYS],edgecolor="k")
    ax1.set_title("Time",fontweight="bold")
    ax1.set_xticks(x)
    ax1.set_xticklabels(METHOD_LABELS,rotation=45,ha="right")
    ax2.bar(x,mms,color=[COLORS[m] for m in METHOD_KEYS],edgecolor="k")
    ax2.set_title("GPU Mem",fontweight="bold")
    ax2.set_xticks(x)
    ax2.set_xticklabels(METHOD_LABELS,rotation=45,ha="right")
    fig.tight_layout()
    savefig(fig,"fig8_training_cost")

def fig9_total_loss_convergence(all_losses):
    print("\n[Fig9] 总损失收敛...")
    fig=plt.figure(figsize=(WIDTH_FULL,4.5))
    gs=GridSpec(2,2,figure=fig,hspace=0.35,wspace=0.3)
    for i,pde in enumerate(PDE_LIST):
        ax=fig.add_subplot(gs[i//2,i%2])
        for m in METHOD_KEYS:
            try:
                y=smooth(all_losses[pde][m]["Ltot"])
                ax.semilogy(y/y[0],**style_for_method(m))
            except: continue
        plot_warmup(ax)
        ax.set_title(PDE_TITLES[i],fontweight="bold")
        ax.set_ylabel(r"$L/L_0$")
        add_panel_label(ax,f"({chr(97+i)})")
    savefig(fig,"fig9_total_loss_convergence")

# ==============================================================================
# 🔥 【完整版】所有方法 所有指标 完整输出（论文直接用）
# ==============================================================================
def analyze_paper_results(table):
    print("\n" + "="*110)
    print("📌 【完整论文数据】所有方法 | 所有方程 | 所有评价指标（精确可复制）")
    print("="*110)

    METRICS = [
        ("mean_l2_raw", "L2 误差"),
        ("mean_rel_l2_raw", "相对L2"),
        ("mean_linf_raw", "Linf 误差"),
        ("mean_constraint_raw", "约束违反"),
        ("mean_residual_raw", "PDE 残差"),
        ("train_time_mean", "时间(s)"),
        ("peak_gpu_mean", "显存(GB)"),
    ]

    for pde in PDE_LIST:
        if pde not in table: continue
        print(f"\n✅ {pde} 方程 —— 全部方法完整指标")
        print("-" * 110)
        print(f"{'方法':<16} {'L2':<11} {'相对L2':<11} {'Linf':<11} {'约束':<11} {'残差':<12} {'时间':<9} {'显存':<9}")
        print("-" * 110)

        for m in METHOD_KEYS:
            if m not in table[pde]: continue
            d = table[pde][m]
            l2 = d.get("mean_l2_raw", 0)
            rl2 = d.get("mean_rel_l2_raw", 0)
            linf = d.get("mean_linf_raw", 0)
            cons = d.get("mean_constraint_raw", 0)
            res = d.get("mean_residual_raw", 0)
            time = d.get("train_time_mean", 0)
            gpu = d.get("peak_gpu_mean", 0)

            print(f"{m:<16} {l2:<11.4f} {rl2:<11.4f} {linf:<11.4f} {cons:<11.4f} {res:<12.4e} {time:<9.1f} {gpu:<9.3f}")

        print("-" * 110)

    print("\n" + "="*70)
    print("📌 核心提升（直接写论文）")
    print("="*70)
    for pde in PDE_LIST:
        if pde not in table or OURS not in table[pde] or "Vanilla" not in table[pde]:
            continue
        v = table[pde]["Vanilla"]
        o = table[pde][OURS]
        imp_c = (v["mean_constraint_raw"] - o["mean_constraint_raw"]) / v["mean_constraint_raw"] * 100
        imp_t = (v["train_time_mean"] - o["train_time_mean"]) / v["train_time_mean"] * 100
        print(f"{pde:<8} → 约束降低 {imp_c:>5.1f}%  |  训练加速 {imp_t:>5.1f}%")

# ==============================================================================
# MAIN
# ==============================================================================
def main():
    print("="*60)
    print("  JCP 可视化脚本（已修复 · 可直接运行）")
    print("="*60)
    table = load_json("ablation_table.json")
    if not table:
        print("❌ 请先运行主实验")
        return

    all_losses = load_all_loss_data()
    fig1_conservation()
    fig2_pareto_front(table)
    fig3_projection_l2(table)
    fig4_projection_residual(table)
    fig5_projection_params(table)
    fig6_training_loss_kdv(all_losses)
    fig8_training_cost(table)
    fig9_total_loss_convergence(all_losses)

    # 自动输出完整论文数据
    analyze_paper_results(table)

    print("\n✅ 全部完成！图表在：", FIG_DIR)

if __name__ == "__main__":
    main()

  JCP 可视化脚本（已修复 · 可直接运行）

[Fig 1] 绘制守恒性对比...
  ✓ fig1_conservation_main.pdf/.png/.eps

[Fig 2] 绘制Pareto前沿...
  ✓ fig2_pareto_frontier.pdf/.png/.eps

[Fig3] 投影L2对比...
  ✓ fig3_projection_l2.pdf/.png/.eps

[Fig4] 投影残差对比...
  ✓ fig4_projection_residual.pdf/.png/.eps

[Fig5] 投影参数...
  ✓ fig5_projection_params.pdf/.png/.eps

[Fig6] KdV损失...
  ✓ fig6_training_loss_kdv.pdf/.png/.eps

[Fig8] 训练成本...
  ✓ fig8_training_cost.pdf/.png/.eps

[Fig9] 总损失收敛...
  ✓ fig9_total_loss_convergence.pdf/.png/.eps

📌 【完整论文数据】所有方法 | 所有方程 | 所有评价指标（精确可复制）

✅ KdV 方程 —— 全部方法完整指标
--------------------------------------------------------------------------------------------------------------
方法               L2          相对L2        Linf        约束          残差           时间        显存       
--------------------------------------------------------------------------------------------------------------
Vanilla          0.0381      0.2086      0.1776      0.0173      2.0224e-05   269.8     0.241    
Penalty          0.0380   